<div align="center">
  <img src="assets/alwer.jpeg" width="140" alt="Alwer logo" />
</div>


<div align="center">

# Machine Learning in Python

## Mini Project 3 — Titanic Survival Prediction

</div>

**Dataset:** [Kaggle Titanic Competition](https://www.kaggle.com/c/titanic)  
**Goal:** predict whether a passenger survived (`Survived = 1`) or died (`Survived = 0`).  
**Models compared:** Logistic Regression, KNN, Decision Tree, Random Forest, SVM, XGBoost  
**Best result:** XGBoost — 83.24% cross-validation accuracy


<img src = "https://storage.googleapis.com/kaggle-competitions/kaggle/3136/logos/front_page.png" width=50%>

## Importing the libraries

### Step 1 · Libraries | گام ۱ — کتابخانه‌ها


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC

## Load and Prepare Data

### Step 2 · Load Data | گام ۲ — بارگذاری داده


In [ ]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

## EDA

### Step 3a · Shape | گام ۳ — ابعاد داده


In [ ]:
print("train:", train.shape)
print("test: ", test.shape)

### Step 3b · Train Head | گام ۳ — پیش‌نمایش train


In [ ]:
print(train.head())

### Step 3c · Test Head | گام ۳ — پیش‌نمایش test


In [ ]:
print(test.head())

### Step 3d · Data Types | گام ۳ — نوع داده‌ها


In [ ]:
train.info()

### Step 3e · Missing Values | گام ۳ — مقادیر گمشده


In [ ]:
print(train.isna().sum())

### Step 3f · Target Balance | گام ۳ — توزیع هدف


In [ ]:
print(train["Survived"].value_counts())

### Step 3g · Describe | گام ۳ — آمار توصیفی


In [ ]:
print(train.describe())

### Step 3h · Categorical Counts | گام ۳ — شمارش دسته‌ای


In [ ]:
for col in ["Sex", "Pclass", "Embarked", "SibSp", "Parch"]:
    print(f"\n=== {col} ===")
    print(train[col].value_counts())
    print("null:", train[col].isna().sum())

### Step 4a · Age and Fare | گام ۴ — نمودار سن و بلیت


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(data=train, x="Age", hue="Survived", bins=30, multiple="stack", kde=True, ax=axes[0])
axes[0].set_title("Age Distribution by Survival")
axes[0].set_xticks(range(0, 81, 5))

sns.histplot(data=train, x="Fare", hue="Survived", bins=30, multiple="stack", kde=True, ax=axes[1])
axes[1].set_title("Fare Distribution by Survival")

plt.tight_layout()
plt.show()

### Step 4b · Correlation Heatmap | گام ۴ — هیت‌مپ همبستگی


In [ ]:
num_only = train.select_dtypes("number").drop(columns=["Survived"])

plt.figure(figsize=(10, 8))
sns.heatmap(num_only.corr(), annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Feature Correlation")
plt.tight_layout()
plt.show()

### Step 4c · Correlation with Target | گام ۴ — همبستگی با هدف


In [ ]:
corr_with_target = num_only.corrwith(train["Survived"]).sort_values()

plt.figure(figsize=(10, 6))
corr_with_target.plot(kind="bar", color="steelblue")
plt.axhline(0, color="red", linestyle="--")
plt.title("Correlation with Survived")
plt.xlabel("Feature")
plt.ylabel("Correlation")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Data Preprocessing

### Step 5a · FamilySize | گام ۵ — اندازه خانواده


In [ ]:
train["FamilySize"] = train["SibSp"] + train["Parch"] + 1
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

### Step 5b · IsAlone | گام ۵ — تنها بودن


In [ ]:
train["IsAlone"] = (train["FamilySize"] == 1).astype(int)
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

### Step 5c · Extract Title from Name | گام ۵ — استخراج عنوان از نام


In [ ]:
# The title reveals gender, age and marital status.
# عنوان از نام، جنسیت، سن و وضعیت تأهل را لو می‌دهد.
#   Mr.     -> adult male
#   Mrs.    -> adult married female
#   Miss.   -> young female
#   Master. -> young boy (children had priority seats)
#
# Must run BEFORE we drop the Name column.
# باید قبل از حذف ستون Name اجرا شود.

for df in (train, test):
    df["Title"] = df["Name"].str.extract(r"([A-Za-z]+)\.", expand=False)

print(train["Title"].value_counts())

### Step 5d · Drop Useless Columns | گام ۵ — حذف ستون‌های بی‌فایده


In [ ]:
test_ids = test["PassengerId"].copy()

# Name and Ticket are kept for now, because the next-next cell still needs
# Name to build the Title column. They are dropped when X is built.
# ستون‌های Name و Ticket فعلاً نگه داشته می‌شوند چون برای ساخت Title لازم‌اند.
useless = ["PassengerId", "Cabin"]
train.drop(columns=useless, inplace=True)
test.drop(columns=useless, inplace=True)

print(train.shape, test.shape)
print(train.columns.tolist())

### Step 5e · Handle Missing Values | گام ۵ — مدیریت مقادیر گمشده


In [ ]:
age_median = train["Age"].median()
train["Age"] = train["Age"].fillna(age_median)
test["Age"] = test["Age"].fillna(age_median)

train = train.dropna(subset=["Embarked"])
test["Embarked"] = test["Embarked"].fillna(test["Embarked"].mode()[0])

fare_median = train["Fare"].median()
train["Fare"] = train["Fare"].fillna(fare_median)
test["Fare"] = test["Fare"].fillna(fare_median)

print("train nulls:", train.isna().sum().sum())
print("test  nulls:", test.isna().sum().sum())
print("train:", train.shape, " test:", test.shape)

### Step 5f · Encode Categories | گام ۵ — کدگذاری دسته‌ها


In [ ]:
train = pd.get_dummies(train, columns=["Embarked"])
test = pd.get_dummies(test, columns=["Embarked"])

train["Sex"] = LabelEncoder().fit_transform(train["Sex"])
test["Sex"] = LabelEncoder().fit_transform(test["Sex"])

print("train:", train.shape, "test:", test.shape)
print(train.columns.tolist())
print()
print("no object left:", [c for c in train.columns if train[c].dtype == "object"])

### Step 5g · Binning | گام ۵ — گروه‌بندی مقادیر


In [ ]:
def group_age(age):
    if age <= 5:  return 1
    if age <= 12: return 2
    if age <= 20: return 3
    if age <= 40: return 4
    if age <= 60: return 5
    return 6

def group_family(n):
    if n == 1: return 1
    if n == 2: return 2
    if n <= 4: return 3
    return 4

for df in (train, test):
    df["AgeGroup"] = df["Age"].apply(group_age)
    df["FamilyGroup"] = df["FamilySize"].apply(group_family)

print(train["AgeGroup"].value_counts().sort_index())
print(train["FamilyGroup"].value_counts().sort_index())

### Step 5h · Build X and y | گام ۵ — ساخت ورودی و هدف


In [ ]:
# Drop the raw text columns now that Title already captured their signal.
# حالا ستون‌های متنی را حذف می‌کنیم چون Title قبلاً سیگنالشان را گرفته.

X = train.drop(columns=["Survived", "Name", "Ticket"])
y = train["Survived"]

test = test.drop(columns=["Name", "Ticket"])

# Train Title is now a category, so encode it as one-hot.
# ستون Title حالا دسته‌ای است، پس به‌صورت one-hot کدگذاری می‌شود.
X = pd.get_dummies(X, columns=["Title"])
test = pd.get_dummies(test, columns=["Title"])

# Align columns so train and test have identical features.
# ستون‌ها را هماهنگ می‌کنیم تا train و test ویژگی‌های یکسان داشته باشند.
test = test.reindex(columns=X.columns, fill_value=0)

# bool must become int so the models accept it.
# نوع bool باید به int تبدیل شود تا مدل‌ها آن را بپذیرند.
for col in X.columns:
    if X[col].dtype == "bool":
        X[col] = X[col].astype(int)
        test[col] = test[col].astype(int)

print("X:", X.shape, "y:", y.shape, "test:", test.shape)
print("cols match:", list(X.columns) == list(test.columns))
print("object left:", [k for k in X.columns if X[k].dtype == "object"])

### Step 6a · Survival Rate by Group | گام ۶ — نرخ بقا


In [ ]:
pretty = {"Sex": {0: "female", 1: "male"},
          "Pclass": {1: "1st", 2: "2nd", 3: "3rd"},
          "IsAlone": {0: "with family", 1: "alone"}}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, ["Sex", "Pclass", "IsAlone"]):
    rate = train.groupby(col)["Survived"].mean() * 100
    rate.index = [pretty[col].get(v, str(v)) for v in rate.index]
    bars = rate.plot(kind="bar", ax=ax, title=f"Survival Rate by {col}", color="steelblue")
    for b, v in zip(bars.patches, rate.values):
        ax.text(b.get_x() + b.get_width()/2, v + 1, f"{v:.1f}%", ha="center")
    ax.set_ylabel("Survival Rate (%)")
    ax.set_xlabel("")
    ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

### 🚢 Storytelling: What Determined the Survival of the Passengers? | چه چیزی سرنوشت مسافران را تعیین کرد؟

**Context | زمینه:**  
In April 1912 the Titanic sank and only 718 of about 2200 passengers survived. The question is: which factors determined the survival rate?  
در آوریل ۱۹۱۲ کشتی تایتانیک غرق شد و تنها ۷۱۸ نفر از حدود ۲۲۰۰ مسافر زنده ماندند. سؤال این است: چه عواملی نرخ بقا را تعیین کردند؟

**Findings | یافته‌ها:**  
- Women survived at 74%, while men only at 19%.  
  زن‌ها ۷۴٪ زنده ماندند، مردها فقط ۱۹٪.
- First class passengers survived at 63%, third class only at 24%.  
  مسافران طبقهٔ اول ۶۳٪ زنده ماندند، طبقهٔ سوم فقط ۲۴٪.
- Passengers travelling with family survived at 51%, those alone at only 30%.  
  مسافرانی که با خانواده بودند ۵۱٪ زنده ماندند، تنهاها فقط ۳۰٪.

**Meaning | معنی:**  
In a crisis, people survive when they have **legal priority** (women and children), or **money** (upper class), or **a support network** (family).  
در بحران، کسانی نجات پیدا می‌کنند که **اولویت قانونی** داشته باشند (زن‌ها و کودکان)، یا **توان مالی** (طبقهٔ بالا)، یا **پشتیبان** (خانواده).

**Model connection | ارتباط با مدل:**  
These three factors are exactly the strongest features the model relies on, which is why it reaches 83% accuracy.  
این سه عامل دقیقاً همان ویژگی‌هایی هستند که مدل روی آن‌ها تکیه می‌کند، و به همین دلیل به ۸۳٪ دقت می‌رسد.


### Step 6b · Distribution Comparison | گام ۶ — مقایسه توزیع‌ها


In [ ]:
plot_df = train.copy()
plot_df["Sex"] = plot_df["Sex"].map({0: "female", 1: "male"})
plot_df["Pclass"] = plot_df["Pclass"].map({1: "1st", 2: "2nd", 3: "3rd"})
plot_df["Survived"] = plot_df["Survived"].map({0: "died", 1: "survived"})

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, col in zip(axes.flatten(), ["Age", "Fare", "Sex", "Pclass"]):
    for name, color in [("died", "#e15759"), ("survived", "#4c9f70")]:
        sub = plot_df[plot_df["Survived"] == name]
        ax.hist(sub[col], bins=25, alpha=0.55, label=name, color=color, density=True)
    ax.set_xlabel(col)
    ax.set_ylabel("Density")
    ax.legend(title="Survived", fontsize=9)

fig.suptitle("Feature Distributions by Survival", fontsize=15)
plt.tight_layout()
plt.show()

## Train your model (Classification)

### Step 7a · Train Test Split | گام ۷ — تقسیم داده


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("survive rate train:", round(y_train.mean(), 3))
print("survive rate test: ", round(y_test.mean(), 3))

### Step 7b · Feature Scaling | گام ۷ — نرمال‌سازی ویژگی‌ها


In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("before:", X_train[["Age", "Fare"]].mean().round(1).to_dict())
print("after: ", X_train_scaled.mean(axis=0).round(1))
print("std:   ", X_train_scaled.std(axis=0).round(1))

### Step 7c · Logistic Regression | گام ۷ — اولین مدل


In [ ]:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("AUC:     ", round(roc_auc_score(y_test, y_prob), 4))
print()
print(classification_report(y_test, y_pred))

### Step 7d · Compare Six Models | گام ۷ — مقایسه شش مدل


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(),
    "KNN":                 KNeighborsClassifier(n_neighbors=5),
    "Decision Tree":       DecisionTreeClassifier(random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM":                 SVC(probability=True, random_state=42),
    "XGBoost":             XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42, eval_metric="logloss"),
}

results = {}

for name, m in models.items():
    m.fit(X_train_scaled, y_train)
    pred = m.predict(X_test_scaled)
    prob = m.predict_proba(X_test_scaled)[:, 1]
    results[name] = {
        "accuracy": accuracy_score(y_test, pred),
        "auc": roc_auc_score(y_test, prob),
    }
    print(f"{name:22} acc={results[name]['accuracy']:.4f}  auc={results[name]['auc']:.4f}")

### Step 7e · Cross Validation | گام ۷ — اعتبارسنجی متقاطع


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_results = {}

for name, m in models.items():
    scores = cross_val_score(m, X, y, cv=cv, scoring="accuracy")
    cv_results[name] = {
        "cv_mean": scores.mean(),
        "cv_std": scores.std(),
        "single_split": results[name]["accuracy"],
    }
    print(f"{name:22} cv={scores.mean():.4f} (+/- {scores.std():.4f})   single={results[name]['accuracy']:.4f}")

print()
print("fold scores for Logistic Regression:")
lr_scores = cross_val_score(models["Logistic Regression"], X, y, cv=cv, scoring="accuracy")
for i, s in enumerate(lr_scores, 1):
    print(f"  fold {i}: {s:.4f}")

## Test Result

### Step 8 · Tune Hyperparameters | گام ۸ — تنظیم پارامترها


In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])

param_grid = {
    "clf__C": [0.01, 0.1, 1, 10, 100],
    "clf__penalty": ["l1", "l2"],
    "clf__solver": ["liblinear", "saga"],
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy",
    n_jobs=-1,
)

grid.fit(X, y)

print("best params:", grid.best_params_)
print("best cv acc:", round(grid.best_score_, 4))
print()
top = pd.DataFrame(grid.cv_results_["params"]).assign(
    mean=grid.cv_results_["mean_test_score"],
    std=grid.cv_results_["std_test_score"],
)
print(top.sort_values("mean", ascending=False).head(8).to_string(index=False))

### Step 9 · Model Comparison Table | گام ۹ — جدول مقایسه مدل‌ها


In [ ]:
comparison = pd.DataFrame(cv_results).T.sort_values("cv_mean", ascending=False)

for col in ["cv_mean", "cv_std", "single_split"]:
    comparison[col] = (comparison[col] * 100).round(2)

comparison = comparison.rename(columns={
    "cv_mean": "CV Accuracy (%)",
    "cv_std": "CV Std (%)",
    "single_split": "Single Split (%)",
})

print(comparison.to_string())
print()
baseline = y.value_counts(normalize=True).max()
print(f"Baseline (always majority class): {baseline * 100:.2f}%")
print()
winner = comparison.index[0]
print(f"Best model by CV: {winner}")
comparison.to_csv("model_comparison.csv")
print("saved -> model_comparison.csv")

### Step 10 · Build Submission File | گام ۱۰ — ساخت فایل خروجی


In [ ]:
submission = pd.DataFrame({
    "PassengerId": test_ids,
    "Survived": 0,
})

print("shape:", submission.shape)
print("cols:", submission.columns.tolist())

### Step 11 · Tune the Best Model | گام ۱۱ — تنظیم بهترین مدل


In [ ]:
xgb_pipe = Pipeline([
    ("clf", XGBClassifier(eval_metric="logloss", random_state=42)),
])

xgb_grid = {
    "clf__n_estimators": [100, 300],
    "clf__max_depth": [2, 3, 4],
    "clf__learning_rate": [0.05, 0.1, 0.3],
    "clf__subsample": [0.8, 1.0],
}

xgb_search = GridSearchCV(
    xgb_pipe,
    xgb_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="accuracy",
    n_jobs=-1,
)

xgb_search.fit(X, y)

print("best params:", xgb_search.best_params_)
print("best cv acc:", round(xgb_search.best_score_, 4))

### Step 12 · Final Prediction | گام ۱۲ — پیش‌بینی نهایی


In [ ]:
best_model = xgb_search.best_estimator_
best_model.fit(X, y)

submission["Survived"] = best_model.predict(test)

print(submission.head())
print()
print("survived:", int(submission["Survived"].sum()), "of", len(submission))
print("mean rate:", round(submission["Survived"].mean(), 4), "vs train", round(y.mean(), 4))

submission.to_csv("submission.csv", index=False)
print("saved -> submission.csv")

## Submit Your Result
